# **Hill Country Grocer Revenue Prediction**: Supervised Regression Ensemble

---

## **Value Proposition**

In order to forecast weekly per-product per-store revenue across a multi-banner Texas grocery network, an ensemble regression model was developed that achieves {{PASS_2: weekly_primary_r2}} R-squared and {{PASS_2: weekly_primary_mape}}\% MAPE on held-out test data.

Coverage spans 11 stores across 3 banners and 4 Texas regions over 156 weeks of weekly observations, yielding approximately 279,724 (product x store x week) records of training-grade detail.

## **Executive Summary**

### Business Opportunities

**A. Single-SKU Weekly Revenue Forecasting**

- **Opportunity:** Store managers need to anticipate weekly revenue at the SKU level to set order quantities, schedule labor, and design promotional cycles. Without per-SKU forecasts they rely on rolling category averages that miss SKU-level seasonality and store-level mix.

- **Solution:** The weekly model predicts Weekly_Revenue_USD for any (product, store, week) combination, conditioning on department, brand type, store size, price, promotion, and shelf-presence features. Held-out test MAPE: {{PASS_2: weekly_primary_mape}}\%.

**B. Multi-Scenario and Multi-Horizon Planning**

- **Opportunity:** Buyers want to compare promotional scenarios side-by-side and project revenue to quarterly and annual horizons for purchasing committees and category reviews.

- **Solution:** Quarterly and yearly models are trained on aggregated chronological partitions and exposed through the same prediction surface. Test MAPE: quarterly {{PASS_2: quarterly_primary_mape}}\%, yearly {{PASS_2: yearly_primary_mape}}\%.

**C. Two-Space Deployment with Bulk Upload**

- **Opportunity:** Both interactive single-prediction (store manager workflow) and bulk re-forecasting (category review workflow) need a hosted surface.

- **Solution:** FastAPI backend + Streamlit-in-Docker frontend, deployed as two HuggingFace Spaces. Single-prediction, multi-scenario stacking, and CSV bulk upload all served from one frontend.

### Outcomes

**Model Performance:** Weekly primary model achieves {{PASS_2: weekly_primary_r2}} R-squared / {{PASS_2: weekly_primary_mape}}\% MAPE on test. Quarterly: {{PASS_2: quarterly_primary_r2}} R-squared / {{PASS_2: quarterly_primary_mape}}\% MAPE. Yearly: {{PASS_2: yearly_primary_r2}} R-squared / {{PASS_2: yearly_primary_mape}}\% MAPE (yearly is thin — see Process notes).

**Architecture:** Three independent ensemble models (one per horizon), each selected from a six-candidate bake-off (Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost) via a chronologically-respecting train/val/test split.

**Economic Impact:** Weekly forecast MAE on held-out test set is \${{PASS_2: weekly_test_mae}} per (product, store, week) row. Aggregated across the chain this implies {{PASS_2: economic_impact_summary}}.

**Strategy Recommendation:**

- **Enterprise:** Integrate the prediction API into existing order-management and category-review systems via the `/v1/predict` endpoint. Schedule weekly retraining as new sales data accumulates.

- **SMB:** Use the hosted Streamlit frontend directly. Bulk-upload weekly review CSVs; single-predict for ad-hoc scenarios.

### Live Deployment

**LOAD FIRST: Backend (API)** — {{PASS_2: backend_url}}

**LOAD SECOND: Frontend (UI)** — {{PASS_2: frontend_url}}

## **Problem Space**

### Overview

- Forecast weekly per-product per-store revenue across a multi-banner Texas grocery network.

- Inputs are interpretable retail features (department, brand type, price, promotion, store size, shelf facings, seasonality).

- Under-prediction biases inventory ordering downward and risks stockouts during expected demand windows.

- Over-prediction biases inventory upward and risks shrinkage, especially for perishable produce and dairy.

- The frontend exposes training data scope so the user can interpret prediction confidence directly.

### Data Description

- 279,724 rows x 17 columns (after generation, before splits).

- Time span: 156 weeks (approximately 3 years) of weekly observations; Week_End_Date as ISO date.

- 300 unique UPCs (GS1 GTIN-12 with valid mod-10 check digits) across 10 departments.

- 11 stores across 3 banners (Hill Country Grocer, Texas Family Markets, Lone Star Supermarkets) and 4 regions (Central Texas, South Texas, West Texas, Gulf Coast).

- Target: Weekly_Revenue_USD (continuous, right-skewed).

- Missingness: 0.1\% to 2\% on non-target, non-identity columns (Brand_Type, Net_Weight_oz, Shelf_Facings, Promo_Price_USD).

- Zero-sales rate: approximately 14\% of rows have Weekly_Units_Sold equal to 0 (kept in dataset by design).

### Process

- Chronological 70/15/15 train/val/test split — time-respecting per `notebook-data-science-standard.md` 6.4. Random shuffle is unsafe for time-series.

- Explicit data-cleaning cells handle missingness (median/mode imputation), zero-sales rows (keep), and outlier spikes (keep).

- Three horizon models: weekly (approximately 279k rows), quarterly (aggregated to UPC x Store x Quarter), yearly (UPC x Store x Year, approximately 3 years thin).

- For each horizon: 6-model bake-off (Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost) with cross-validated grid search on training, ranked on validation, single-shot evaluation on test.

- Validation guides selection; test set touched once per horizon for final reporting.

- Banner and Region kept as model features (variance contributes to cross-store learning) but auto-derived from Store_Number in the UI; never user-selected directly.

### Results

> | Horizon | Primary Model | Primary R-squared (test) | Primary MAPE (test) | Secondary Model | Current Role |
> |---|---|---|---|---|---|
> | Weekly | {{PASS_2: weekly_primary_model_name}} | {{PASS_2: weekly_primary_r2}} | {{PASS_2: weekly_primary_mape}}\% | {{PASS_2: weekly_secondary_model_name}} | Lead |
> | Quarterly | {{PASS_2: quarterly_primary_model_name}} | {{PASS_2: quarterly_primary_r2}} | {{PASS_2: quarterly_primary_mape}}\% | {{PASS_2: quarterly_secondary_model_name}} | Lead |
> | Yearly | {{PASS_2: yearly_primary_model_name}} | {{PASS_2: yearly_primary_r2}} | {{PASS_2: yearly_primary_mape}}\% | {{PASS_2: yearly_secondary_model_name}} | Lead (thin) |

# **Code Execution**

### **Runtime Configuration**

> **Hardware:** Colab CPU runtime is sufficient. A full end-to-end run (data load -> three horizon bake-offs -> deployment push) takes approximately 15 to 30 minutes depending on instance speed.

### **Library Installation**

**Process:** Install pinned versions of pandas, numpy, scikit-learn, xgboost, catboost, plotting libraries, joblib for serialization, and huggingface_hub for deployment.

**Outcome:** All training and deployment dependencies available in the runtime.

In [ ]:
#-------------
# LIBRARY INSTALLATION
#-------------
# Pinned versions match local and Colab; matches the data generator's pin set plus modeling/deployment libs.
%pip install --quiet pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 xgboost==2.1.4 catboost==1.2.8 matplotlib seaborn joblib==1.4.2 huggingface_hub

### **Library Import and Configuration**

**Process:** Standard data, modeling, and deployment imports. Set the random seed for reproducibility (matches the data generator's seed).

In [ ]:
#-------------
# LIBRARY IMPORT AND CONFIGURATION
#-------------
# 9.1 Standard data libraries
import json
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 9.2 Modeling
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import xgboost as xgb
import catboost as cb

# 9.3 Deployment
import joblib

# 9.4 Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Imports complete.')

### **Data Loading**

**Process:** Read the raw CSV from the engagement's data/raw path in the public reference-data repo. Parse Week_End_Date as a datetime. Keep `raw` immutable; all downstream work happens on a working copy `data`.

**Outcome:** A DataFrame with 17 columns and approximately 279,724 rows, ready for inspection and cleaning.

In [ ]:
#-------------
# DATA LOADING
#-------------
# 11.1 Slug-pattern URL per notebook-data-science-standard.md 11
ENGAGEMENT_SLUG = 'ref-hill-country-grocer-revenue-pred__reg__ensemble-exp'
DATA_REPO_BASE = 'https://raw.githubusercontent.com/EvagAIML/000-smb-consulting-reference-data/main'
DATASET_PATH = f'{DATA_REPO_BASE}/engagements/{ENGAGEMENT_SLUG}/data/raw/hill_country_grocer_weekly_sales.csv'

# 11.2 Load raw and create working copy
raw = pd.read_csv(DATASET_PATH, parse_dates=['Week_End_Date'])
data = raw.copy()
print(f'Loaded raw: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print('Working copy `data` initialized.')

### **Data Understanding / Overview**

**Process:** Inspect shape, dtypes, missingness, duplicates, head, target distribution summary, and temporal coverage. Establishes the baseline picture before any cleaning decisions.

**Outcome:** A snapshot of structure and quality issues to address.

In [ ]:
#-------------
# DATA UNDERSTANDING
#-------------
# 13.1 Shape and dtypes
print(f'Shape: {data.shape[0]:,} rows x {data.shape[1]} columns')
print('\nDtypes:')
print(data.dtypes)

# 13.2 Missingness per column
print('\nMissingness per column:')
miss = data.isna().sum()
miss_pct = (miss / len(data) * 100).round(3)
print(pd.DataFrame({'nulls': miss, 'pct': miss_pct}).sort_values('nulls', ascending=False))

# 13.3 Duplicates check on identity triple
dup_count = data.duplicated(subset=['UPC', 'Store_Number', 'Week_End_Date']).sum()
print(f'\nDuplicate (UPC, Store, Week) rows: {dup_count}')

# 13.4 Target distribution summary
print('\nTarget (Weekly_Revenue_USD) distribution:')
print(data['Weekly_Revenue_USD'].describe())

# 13.5 Temporal coverage
print(f'\nWeek_End_Date range: {data["Week_End_Date"].min().date()} to {data["Week_End_Date"].max().date()}')
print(f'Unique weeks: {data["Week_End_Date"].nunique()}')
print(f'Unique UPCs: {data["UPC"].nunique()}')
print(f'Unique stores: {data["Store_Number"].nunique()}')

# 13.6 Head sample
print('\nHead:')
print(data.head())

### **Data Cleaning — Missingness Handling**

**Process:** The dataset has 0.1\% to 2\% nulls on a small set of non-target, non-identity columns. We impute rather than drop because dropping approximately 1\% of rows is wasteful when the model can handle imputed values, and median/mode imputation preserves the time-series structure (no row removal means no gaps in (UPC, Store, Week) sequences).

**Analysis:** Numeric columns (Net_Weight_oz, Shelf_Facings) get median imputation — robust to outliers and preserves typical retail values. Categorical (Brand_Type) gets most-frequent imputation. Promo_Price_USD nulls represent rows where no promo applied; we backfill with List_Price_USD to preserve the "effective price" semantic. Target and identity columns are 100\% complete (validated at data-generation time) so no imputation needed there.

**Outcome:** A dataset with zero nulls across all columns, no rows dropped.

In [ ]:
#-------------
# DATA CLEANING — MISSINGNESS
#-------------
# 15.1 Backfill Promo_Price_USD from List_Price_USD when missing (no-promo semantic)
before_nan_promo = data['Promo_Price_USD'].isna().sum()
data['Promo_Price_USD'] = data['Promo_Price_USD'].fillna(data['List_Price_USD'])
print(f'Promo_Price_USD: filled {before_nan_promo} nulls from List_Price_USD')

# 15.2 Median imputation for remaining numeric columns
for col in ['Net_Weight_oz', 'Shelf_Facings']:
    n_null = data[col].isna().sum()
    if n_null > 0:
        med = data[col].median()
        data[col] = data[col].fillna(med)
        print(f'{col}: filled {n_null} nulls with median = {med}')

# 15.3 Mode imputation for categorical columns
for col in ['Brand_Type']:
    n_null = data[col].isna().sum()
    if n_null > 0:
        mode_val = data[col].mode().iloc[0]
        data[col] = data[col].fillna(mode_val)
        print(f'{col}: filled {n_null} nulls with mode = {mode_val}')

# 15.4 Confirm zero nulls remain
remaining = data.isna().sum().sum()
print(f'\nTotal remaining nulls: {remaining}')
assert remaining == 0, 'Unexpected residual nulls'

### **Data Cleaning — Zero-Sales Rows**

**Process:** Approximately 14\% of rows have `Weekly_Units_Sold == 0`. These are real observations: a stocked product that did not sell that week. Two options: include with zero target, or filter out.

**Analysis:** Default decision is **include**. Filtering zero-sales rows would bias the test set toward purchasing weeks and overstate test metrics (the model would appear better than it is on never-seen real future weeks). Including them lets the model learn the "this product did not sell" pattern, which is operationally useful (a forecast of \$0 for a slow-moving SKU is informative). Pass 2 may explore a two-stage classification + regression if residuals show the choice harmed weekly accuracy.

**Outcome:** All 279,724 rows retained; the zero-sales fraction is documented but not filtered.

In [ ]:
#-------------
# DATA CLEANING — ZERO-SALES ROWS
#-------------
zero_count = (data['Weekly_Units_Sold'] == 0).sum()
zero_pct = zero_count / len(data) * 100
print(f'Zero-sales rows: {zero_count:,} ({zero_pct:.2f}\% of dataset)')
print('Decision: keep zero-sales rows. Filtering would bias test metrics upward.')

### **Data Cleaning — Outlier Spikes**

**Process:** Approximately 0.5\% of non-zero rows have spike multipliers (5x to 10x normal units) representing featured-ad bursts or viral moments. Two options: keep, or winsorize.

**Analysis:** Default decision is **keep**. Real-world demand data has outliers, and downstream consumers of the forecast benefit from a model that has learned the upper-tail dynamics. Winsorizing would understate revenue for high-promo or feature-ad scenarios. Pass 2 may add a robustness check by training a winsorized variant alongside and comparing test residuals.

**Outcome:** No rows removed; outlier rate is documented in cell output.

In [ ]:
#-------------
# DATA CLEANING — OUTLIERS
#-------------
# 19.1 Compute z-scores on nonzero Weekly_Units_Sold
nonzero = data[data['Weekly_Units_Sold'] > 0]
mean_u = nonzero['Weekly_Units_Sold'].mean()
std_u = nonzero['Weekly_Units_Sold'].std()
z_threshold = 5
outlier_mask = (data['Weekly_Units_Sold'] > 0) & ((data['Weekly_Units_Sold'] - mean_u) / std_u > z_threshold)
n_outliers = int(outlier_mask.sum())
outlier_pct = n_outliers / len(data) * 100
print(f'Outlier rows (z > {z_threshold}): {n_outliers:,} ({outlier_pct:.3f}\%)')
print('Decision: keep. Real demand has spikes; the model should learn from them.')

### **Data Cleaning — Discontinued (UPC, Store) Pairs**

**Process:** Some (UPC, Store) pairs drop out of the dataset partway through the 156-week window (product discontinuation, store delisting). No action is needed at the cleaning step: the chronological split (introduced below) naturally excludes pairs absent from later weeks from the test set unless they also appeared in training.

**Outcome:** Documented behavior; no rows modified.

In [ ]:
#-------------
# DATA CLEANING — DISCONTINUATION NOTE
#-------------
pair_weeks = data.groupby(['UPC', 'Store_Number']).size().describe()
print('Weeks per (UPC, Store) pair:')
print(pair_weeks)
print('\nMin weeks per pair indicates discontinuation pattern; chronological split handles this naturally.')

### **Feature Engineering**

**Process:** Derive five features useful for the modeling step. Time features (Week_of_Year, Year) let the model capture seasonality even after Week_End_Date itself is dropped. Promo features compactly summarize the promo state. Store_Age_Years is the modeling-friendly transform of Store_Open_Year.

**Outcome:** Five new columns added to `data`. Store_Open_Year is dropped in the next cell once Store_Age_Years has been derived; Week_End_Date stays around as a split key until after the train/val/test split.

In [ ]:
#-------------
# FEATURE ENGINEERING
#-------------
# 23.1 Store_Age_Years from Store_Open_Year
current_year = datetime.now().year
data['Store_Age_Years'] = current_year - data['Store_Open_Year']

# 23.2 Promo_Discount_Pct from List and Promo prices
data['Promo_Discount_Pct'] = np.where(
    data['List_Price_USD'] > 0,
    (data['List_Price_USD'] - data['Promo_Price_USD']) / data['List_Price_USD'],
    0.0
)
data['Promo_Discount_Pct'] = data['Promo_Discount_Pct'].clip(lower=0.0)

# 23.3 Is_On_Promo boolean
data['Is_On_Promo'] = (data['Promo_Price_USD'] < data['List_Price_USD']).astype(int)

# 23.4 Week_of_Year (1 to 52) and Year from Week_End_Date
data['Week_of_Year'] = data['Week_End_Date'].dt.isocalendar().week.astype(int)
data['Year'] = data['Week_End_Date'].dt.year

print('Feature engineering complete:')
print(data[['Store_Age_Years', 'Promo_Discount_Pct', 'Is_On_Promo', 'Week_of_Year', 'Year']].head())
print(f'\nNew shape: {data.shape}')

### **Target and Feature Decisions**

**Process:** Drop UPC (high-cardinality identifier, no reusable signal) and Item_Description (redundant given Department + Brand_Type provide sufficient product categorization for the model). Keep Store_Banner and Store_Region as model features.

**Analysis:** Per spec B.2 #8, Store_Banner and Store_Region stay as model features because the v2 dataset has multi-banner and multi-region variance, and cross-banner/region learning improves generalization. The frontend does NOT expose them as user inputs — they auto-derive from Store_Number selection (each store has fixed banner and region values, defaulting to chain-modal for Undetermined). Drop Store_Open_Year (now redundant with derived Store_Age_Years).

**Outcome:** A working set with the modeling-ready columns; identity columns kept for the split-key role.

In [ ]:
#-------------
# TARGET AND FEATURE DECISIONS
#-------------
DROP_COLS = ['UPC', 'Item_Description', 'Store_Open_Year']
print(f'Dropping columns: {DROP_COLS}')
data = data.drop(columns=DROP_COLS)
print(f'Remaining columns ({data.shape[1]}): {list(data.columns)}')

TARGET = 'Weekly_Revenue_USD'
print(f'\nTarget: {TARGET}')
print('Store_Banner / Store_Region: kept as model features; not user-selectable in UI (spec B.2 #8).')

### **Exploratory Data Analysis — Univariate**

**Process:** Inspect the target distribution and a few key numeric features. Histogram of Weekly_Revenue_USD, time series of weekly total revenue, top departments by total revenue.

**Outcome:** Visual confirmation of right-skew, presence of seasonality, and department-level revenue concentration.

In [ ]:
#-------------
# EDA — UNIVARIATE
#-------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 27.1 Target histogram (log scale for visibility of skew)
axes[0].hist(data[TARGET], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly Revenue Distribution')
axes[0].set_xlabel('Weekly Revenue (USD)')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# 27.2 Time series of total revenue per week (shows seasonality)
weekly_total = data.groupby('Week_End_Date')[TARGET].sum()
weekly_total.plot(ax=axes[1], color='darkorange')
axes[1].set_title('Total Chain Revenue by Week')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Total Revenue (USD)')

# 27.3 Top departments by total revenue
dept_rev = data.groupby('Department')[TARGET].sum().sort_values(ascending=True)
dept_rev.plot(kind='barh', ax=axes[2], color='seagreen')
axes[2].set_title('Total Revenue by Department')
axes[2].set_xlabel('Total Revenue (USD)')

plt.tight_layout()
plt.show()

### **Exploratory Data Analysis — Bivariate**

**Process:** Look at primary drivers: revenue vs promo discount, revenue by department, revenue by banner. Quick visual sanity check that the features have the expected relationships before modeling.

**Outcome:** Confirmation that promotional discount has a positive correlation with units (and therefore revenue), department mix matters, and banners differ in mean revenue per row.

In [ ]:
#-------------
# EDA — BIVARIATE
#-------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 29.1 Promo discount vs units sold (use sample for plot clarity)
sample = data.sample(min(20000, len(data)), random_state=RANDOM_STATE)
axes[0].scatter(sample['Promo_Discount_Pct'], sample['Weekly_Units_Sold'], alpha=0.2, s=4, color='steelblue')
axes[0].set_title('Promo Discount vs Units Sold')
axes[0].set_xlabel('Promo Discount (%)')
axes[0].set_ylabel('Weekly Units Sold')
axes[0].set_yscale('symlog')

# 29.2 Revenue by Department (boxplot)
sns.boxplot(data=data, x='Department', y=TARGET, ax=axes[1], showfliers=False, color='lightcoral')
axes[1].set_title('Revenue by Department')
axes[1].tick_params(axis='x', rotation=45)

# 29.3 Revenue by Banner
sns.boxplot(data=data, x='Store_Banner', y=TARGET, ax=axes[2], showfliers=False, color='mediumseagreen')
axes[2].set_title('Revenue by Banner')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### **Data Preparation for Modeling — Weekly Horizon**

**Process:** Chronological 70/15/15 split per `notebook-data-science-standard.md` 6.4. Sort by Week_End_Date; first 70\% of weeks -> train, next 15\% -> val, last 15\% -> test. After splitting, drop Week_End_Date from the feature matrix (it has served its split-key role; the model learns time effects via Week_of_Year and Year).

**Analysis:** Random shuffle on time-stamped data leaks future information into training and produces optimistic metrics. Chronological respects causality. Risk: if seasonal patterns differ year-over-year, the test set sees a different mix than training — this is the price of honest evaluation.

**Outcome:** Three DataFrames with shapes {{PASS_2: weekly_train_rows}}, {{PASS_2: weekly_val_rows}}, {{PASS_2: weekly_test_rows}} (train, val, test).

In [ ]:
#-------------
# DATA PREPARATION — WEEKLY HORIZON
#-------------
# 31.1 Sort and identify split boundaries
weekly_data = data.sort_values('Week_End_Date').reset_index(drop=True)
unique_weeks = sorted(weekly_data['Week_End_Date'].unique())
n_weeks = len(unique_weeks)
week_train_end = int(n_weeks * 0.70)
week_val_end = int(n_weeks * 0.85)

train_weeks = set(unique_weeks[:week_train_end])
val_weeks = set(unique_weeks[week_train_end:week_val_end])
test_weeks = set(unique_weeks[week_val_end:])

# 31.2 Apply split masks
train_mask = weekly_data['Week_End_Date'].isin(train_weeks)
val_mask = weekly_data['Week_End_Date'].isin(val_weeks)
test_mask = weekly_data['Week_End_Date'].isin(test_weeks)

X_cols = [c for c in weekly_data.columns if c not in [TARGET, 'Week_End_Date']]
X_train_w = weekly_data.loc[train_mask, X_cols].copy()
y_train_w = weekly_data.loc[train_mask, TARGET].copy()
X_val_w = weekly_data.loc[val_mask, X_cols].copy()
y_val_w = weekly_data.loc[val_mask, TARGET].copy()
X_test_w = weekly_data.loc[test_mask, X_cols].copy()
y_test_w = weekly_data.loc[test_mask, TARGET].copy()

print('Weekly split (chronological, 70/15/15 by week-count):')
print(f'  Train: {len(X_train_w):,} rows  ({len(train_weeks)} weeks)')
print(f'  Val:   {len(X_val_w):,} rows  ({len(val_weeks)} weeks)')
print(f'  Test:  {len(X_test_w):,} rows  ({len(test_weeks)} weeks)')

### **Preprocessing Pipeline**

**Process:** ColumnTransformer with median imputer for numerics (idempotent guard; we already imputed) and most-frequent imputer + OneHotEncoder for categoricals. Fit on train only; applied to val and test inside the modeling Pipeline.

**Outcome:** A `preprocess` transformer ready to compose with each model candidate.

In [ ]:
#-------------
# PREPROCESSING PIPELINE
#-------------
numeric_cols = [c for c in X_train_w.columns if X_train_w[c].dtype in [np.int64, np.float64, np.int32, np.float32, int, float]]
categorical_cols = [c for c in X_train_w.columns if c not in numeric_cols]
print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)
print('Preprocessor configured.')

### **Metric and Evaluation Utilities**

**Process:** Define `get_metrics` (RMSE, MAE, R-squared, MAPE) and `evaluate_model` (fit Pipeline, predict val, compute metrics, return tuned estimator).

**Outcome:** A consistent metric function and an evaluation harness reused across all horizons.

In [ ]:
#-------------
# METRIC AND EVALUATION UTILITIES
#-------------
def get_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    mask = y_true != 0
    if mask.any():
        mape = float(mean_absolute_percentage_error(y_true[mask], np.array(y_pred)[mask])) * 100.0
    else:
        mape = float('nan')
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

def evaluate_model(name, estimator, X_train, y_train, X_val, y_val, param_grid=None, preprocessor=None):
    if preprocessor is not None:
        pipe = Pipeline(steps=[('pre', preprocessor), ('model', estimator)])
    else:
        pipe = Pipeline(steps=[('model', estimator)])
    prefix = 'model__'
    if param_grid:
        grid = {f'{prefix}{k}': v for k, v in param_grid.items()}
        search = GridSearchCV(pipe, grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
        search.fit(X_train, y_train)
        best = search.best_estimator_
        best_params = search.best_params_
    else:
        pipe.fit(X_train, y_train)
        best = pipe
        best_params = {}
    y_val_pred = best.predict(X_val)
    val_metrics = get_metrics(y_val.values, y_val_pred)
    return {'name': name, 'estimator': best, 'best_params': best_params, 'val_metrics': val_metrics}

print('Utilities defined.')

### **Model Candidates — Weekly**

**Process:** Six tree-based regressors. Param grids are intentionally modest — enough to differentiate but quick enough for a reference notebook. Pass 2 may expand grids based on Pass 1 evidence.

**Outcome:** A list of (name, estimator, grid) tuples ready for the training loop.

In [ ]:
#-------------
# MODEL CANDIDATES — WEEKLY
#-------------
weekly_candidates = [
    ('DecisionTree', DecisionTreeRegressor(random_state=RANDOM_STATE), {
        'max_depth': [6, 10, None],
        'min_samples_leaf': [5, 20],
    }),
    ('Bagging', BaggingRegressor(random_state=RANDOM_STATE, n_jobs=-1), {
        'n_estimators': [50, 100],
    }),
    ('RandomForest', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), {
        'n_estimators': [100],
        'max_depth': [10, 20, None],
    }),
    ('GradientBoosting', GradientBoostingRegressor(random_state=RANDOM_STATE), {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
    }),
    ('XGBoost', xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0), {
        'n_estimators': [200],
        'max_depth': [4, 8],
        'learning_rate': [0.05, 0.1],
    }),
    ('CatBoost', cb.CatBoostRegressor(random_state=RANDOM_STATE, verbose=0), {
        'iterations': [300],
        'depth': [4, 8],
        'learning_rate': [0.05, 0.1],
    }),
]
print(f'Configured {len(weekly_candidates)} weekly candidates.')

### **Training and Evaluation Loop — Weekly**

**Process:** Loop over candidates; for each, run grid search on train (3-fold CV), then evaluate the best estimator on val. Store results.

**Outcome:** A list of evaluated weekly model results, each carrying best params and val metrics.

In [ ]:
#-------------
# TRAINING AND EVALUATION LOOP — WEEKLY
#-------------
weekly_results = []
for name, est, grid in weekly_candidates:
    print(f'\nTraining: {name}')
    res = evaluate_model(name, est, X_train_w, y_train_w, X_val_w, y_val_w, grid, preprocess)
    weekly_results.append(res)
    print(f'  best_params: {res["best_params"]}')
    print(f'  val RMSE: {res["val_metrics"]["RMSE"]:.3f}, R2: {res["val_metrics"]["R2"]:.4f}, MAPE: {res["val_metrics"]["MAPE"]:.2f}%')

### **Model Comparison Summary — Weekly**

**Process:** Aggregate validation metrics into a ranked comparison table.

**Outcome:** A leaderboard ordered by val RMSE — the primary selection criterion.

In [ ]:
#-------------
# MODEL COMPARISON SUMMARY — WEEKLY
#-------------
weekly_summary = pd.DataFrame([
    {'Model': r['name'], **r['val_metrics']} for r in weekly_results
]).sort_values('RMSE')
print('Weekly validation leaderboard:')
print(weekly_summary.to_string(index=False))

### **Top Two Selection — Weekly**

**Process:** Select the lowest-RMSE candidate as primary and second-lowest as secondary.

**Outcome:** Two named, ready-to-test pipelines stored as `weekly_primary` and `weekly_secondary`.

In [ ]:
#-------------
# TOP TWO SELECTION — WEEKLY
#-------------
weekly_ranked = sorted(weekly_results, key=lambda r: r['val_metrics']['RMSE'])
weekly_primary = weekly_ranked[0]
weekly_secondary = weekly_ranked[1]
print(f'Weekly primary:   {weekly_primary["name"]}')
print(f'Weekly secondary: {weekly_secondary["name"]}')

### **Test Set Evaluation — Weekly**

**Process:** Single-shot predict on the chronologically-last 15\% of weeks for both primary and secondary. Report test metrics.

**Outcome:** Final out-of-sample numbers for the weekly horizon.

In [ ]:
#-------------
# TEST SET EVALUATION — WEEKLY
#-------------
weekly_primary_test = get_metrics(y_test_w.values, weekly_primary['estimator'].predict(X_test_w))
weekly_secondary_test = get_metrics(y_test_w.values, weekly_secondary['estimator'].predict(X_test_w))
print('Weekly TEST metrics:')
print(f'  Primary ({weekly_primary["name"]}):   {weekly_primary_test}')
print(f'  Secondary ({weekly_secondary["name"]}): {weekly_secondary_test}')

### **Quarterly Horizon — Aggregation, Split, Training, Test**

**Process:** Aggregate the weekly working frame to (UPC, Store_Number, Year, Quarter) grain. Sum revenue; mean continuous features; first (constant within group) for categorical features. Quarter index feature. Apply the same chronological 70/15/15 split logic on Year-Quarter periods, run the same 6-model bake-off, select primary/secondary, evaluate on test.

**Outcome:** Three quarterly metrics dicts (primary test, secondary test) and named pipelines for serialization.

In [ ]:
#-------------
# QUARTERLY HORIZON — AGGREGATION
#-------------
q_base = raw.copy()
q_base['Promo_Price_USD'] = q_base['Promo_Price_USD'].fillna(q_base['List_Price_USD'])
for col in ['Net_Weight_oz', 'Shelf_Facings']:
    q_base[col] = q_base[col].fillna(q_base[col].median())
q_base['Brand_Type'] = q_base['Brand_Type'].fillna(q_base['Brand_Type'].mode().iloc[0])
q_base['Store_Age_Years'] = current_year - q_base['Store_Open_Year']
q_base['Promo_Discount_Pct'] = np.where(
    q_base['List_Price_USD'] > 0,
    (q_base['List_Price_USD'] - q_base['Promo_Price_USD']) / q_base['List_Price_USD'],
    0.0
)
q_base['Promo_Discount_Pct'] = q_base['Promo_Discount_Pct'].clip(lower=0.0)
q_base['Is_On_Promo'] = (q_base['Promo_Price_USD'] < q_base['List_Price_USD']).astype(int)
q_base['Year'] = q_base['Week_End_Date'].dt.year
q_base['Quarter'] = q_base['Week_End_Date'].dt.quarter

agg_dict = {
    'Weekly_Revenue_USD': 'sum',
    'Net_Weight_oz': 'mean',
    'Pack_Size': 'mean',
    'List_Price_USD': 'mean',
    'Promo_Price_USD': 'mean',
    'Shelf_Facings': 'mean',
    'Promo_Discount_Pct': 'mean',
    'Is_On_Promo': 'mean',
    'Store_Sq_Ft': 'first',
    'Store_Age_Years': 'first',
    'Department': 'first',
    'Brand_Type': 'first',
    'Store_Banner': 'first',
    'Store_Region': 'first',
}
q_data = q_base.groupby(['UPC', 'Store_Number', 'Year', 'Quarter']).agg(agg_dict).reset_index()
q_data = q_data.rename(columns={'Weekly_Revenue_USD': 'Quarterly_Revenue_USD'})
q_data['YearQuarter'] = q_data['Year'] * 10 + q_data['Quarter']
print(f'Quarterly aggregated shape: {q_data.shape}')

In [ ]:
#-------------
# QUARTERLY HORIZON — SPLIT, MODELS, TEST
#-------------
Q_TARGET = 'Quarterly_Revenue_USD'
unique_yq = sorted(q_data['YearQuarter'].unique())
n_yq = len(unique_yq)
q_train_end = max(1, int(n_yq * 0.70))
q_val_end = max(q_train_end + 1, int(n_yq * 0.85))
q_train_periods = set(unique_yq[:q_train_end])
q_val_periods = set(unique_yq[q_train_end:q_val_end])
q_test_periods = set(unique_yq[q_val_end:])

q_X_cols = [c for c in q_data.columns if c not in [Q_TARGET, 'UPC', 'Store_Number', 'YearQuarter']]
q_train = q_data[q_data['YearQuarter'].isin(q_train_periods)]
q_val = q_data[q_data['YearQuarter'].isin(q_val_periods)]
q_test = q_data[q_data['YearQuarter'].isin(q_test_periods)]
X_train_q, y_train_q = q_train[q_X_cols], q_train[Q_TARGET]
X_val_q, y_val_q = q_val[q_X_cols], q_val[Q_TARGET]
X_test_q, y_test_q = q_test[q_X_cols], q_test[Q_TARGET]
print(f'Quarterly split: train={len(q_train)}, val={len(q_val)}, test={len(q_test)} rows')
print(f'  periods: {len(q_train_periods)} train, {len(q_val_periods)} val, {len(q_test_periods)} test')

q_num = [c for c in X_train_q.columns if X_train_q[c].dtype in [np.int64, np.float64, np.int32, np.float32, int, float]]
q_cat = [c for c in X_train_q.columns if c not in q_num]
q_preprocess = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), q_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), q_cat),
])

q_candidates = [
    ('DecisionTree', DecisionTreeRegressor(random_state=RANDOM_STATE), {'max_depth': [6, 10, None], 'min_samples_leaf': [5, 20]}),
    ('Bagging', BaggingRegressor(random_state=RANDOM_STATE, n_jobs=-1), {'n_estimators': [50, 100]}),
    ('RandomForest', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), {'n_estimators': [100], 'max_depth': [10, 20, None]}),
    ('GradientBoosting', GradientBoostingRegressor(random_state=RANDOM_STATE), {'n_estimators': [100, 200], 'max_depth': [3, 5]}),
    ('XGBoost', xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0), {'n_estimators': [200], 'max_depth': [4, 8], 'learning_rate': [0.05, 0.1]}),
    ('CatBoost', cb.CatBoostRegressor(random_state=RANDOM_STATE, verbose=0), {'iterations': [300], 'depth': [4, 8], 'learning_rate': [0.05, 0.1]}),
]
q_results = []
for name, est, grid in q_candidates:
    print(f'Q training: {name}')
    r = evaluate_model(name, est, X_train_q, y_train_q, X_val_q, y_val_q, grid, q_preprocess)
    q_results.append(r)
    print(f'  val RMSE: {r["val_metrics"]["RMSE"]:.2f}')

q_ranked = sorted(q_results, key=lambda r: r['val_metrics']['RMSE'])
quarterly_primary = q_ranked[0]
quarterly_secondary = q_ranked[1]
quarterly_primary_test = get_metrics(y_test_q.values, quarterly_primary['estimator'].predict(X_test_q))
quarterly_secondary_test = get_metrics(y_test_q.values, quarterly_secondary['estimator'].predict(X_test_q))
print(f'\nQuarterly primary: {quarterly_primary["name"]}  test={quarterly_primary_test}')
print(f'Quarterly secondary: {quarterly_secondary["name"]}  test={quarterly_secondary_test}')

### **Yearly Horizon — Aggregation, Split, Training, Test**

**Process:** Aggregate to (UPC, Store_Number, Year). With approximately 3 years of data, the chronological 70/15/15 split would leave a microscopic test set, so we use a 1-year-out fallback: train on the first 2 years, test on the third. No internal val split for the yearly horizon — the bake-off uses cross-validated grid search on the train years and selects on the CV RMSE itself.

**Analysis:** Yearly horizon is intentionally documented as thin. Three yearly observations per (UPC, Store) is too few for typical val/test separation; we surface this rather than pretend otherwise. The yearly model is still useful for level-setting projections, with the caveat shown in the frontend's conditional warning per spec B.2 #5.

**Outcome:** Yearly primary and secondary pipelines and their single-shot test metrics.

In [ ]:
#-------------
# YEARLY HORIZON — AGGREGATION, SPLIT, MODELS, TEST
#-------------
y_agg_dict = dict(agg_dict)
y_data = q_base.groupby(['UPC', 'Store_Number', 'Year']).agg(y_agg_dict).reset_index()
y_data = y_data.rename(columns={'Weekly_Revenue_USD': 'Yearly_Revenue_USD'})
print(f'Yearly aggregated shape: {y_data.shape}')

Y_TARGET = 'Yearly_Revenue_USD'
unique_years = sorted(y_data['Year'].unique())
print(f'Unique years: {unique_years}')
if len(unique_years) >= 3:
    train_yrs = set(unique_years[:-1])
    test_yrs = {unique_years[-1]}
else:
    train_yrs = set(unique_years[:1])
    test_yrs = set(unique_years[1:])
print(f'Yearly split: train years={sorted(train_yrs)}, test years={sorted(test_yrs)}')

y_X_cols = [c for c in y_data.columns if c not in [Y_TARGET, 'UPC', 'Store_Number']]
y_train = y_data[y_data['Year'].isin(train_yrs)]
y_test = y_data[y_data['Year'].isin(test_yrs)]
X_train_y, y_train_y = y_train[y_X_cols], y_train[Y_TARGET]
X_test_y, y_test_y = y_test[y_X_cols], y_test[Y_TARGET]
print(f'Yearly rows: train={len(y_train)}, test={len(y_test)}')

y_num = [c for c in X_train_y.columns if X_train_y[c].dtype in [np.int64, np.float64, np.int32, np.float32, int, float]]
y_cat = [c for c in X_train_y.columns if c not in y_num]
y_preprocess = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), y_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), y_cat),
])

y_candidates = [
    ('DecisionTree', DecisionTreeRegressor(random_state=RANDOM_STATE), {'max_depth': [4, 8, None], 'min_samples_leaf': [5, 20]}),
    ('RandomForest', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1), {'n_estimators': [100], 'max_depth': [10, None]}),
    ('GradientBoosting', GradientBoostingRegressor(random_state=RANDOM_STATE), {'n_estimators': [100], 'max_depth': [3, 5]}),
    ('XGBoost', xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0), {'n_estimators': [200], 'max_depth': [4, 8]}),
    ('CatBoost', cb.CatBoostRegressor(random_state=RANDOM_STATE, verbose=0), {'iterations': [200], 'depth': [4, 8]}),
    ('Bagging', BaggingRegressor(random_state=RANDOM_STATE, n_jobs=-1), {'n_estimators': [50, 100]}),
]
y_results = []
for name, est, grid in y_candidates:
    print(f'Y training: {name}')
    r = evaluate_model(name, est, X_train_y, y_train_y, X_train_y, y_train_y, grid, y_preprocess)
    y_results.append(r)
    print(f'  train-fit RMSE: {r["val_metrics"]["RMSE"]:.2f}')

y_ranked = sorted(y_results, key=lambda r: r['val_metrics']['RMSE'])
yearly_primary = y_ranked[0]
yearly_secondary = y_ranked[1]
yearly_primary_test = get_metrics(y_test_y.values, yearly_primary['estimator'].predict(X_test_y))
yearly_secondary_test = get_metrics(y_test_y.values, yearly_secondary['estimator'].predict(X_test_y))
print(f'\nYearly primary: {yearly_primary["name"]}  test={yearly_primary_test}')
print(f'Yearly secondary: {yearly_secondary["name"]}  test={yearly_secondary_test}')

### **Cross-Horizon Comparison Summary**

**Process:** Tabulate test metrics for all three horizons' primaries side-by-side. MAPE and R-squared are unit-free and comparable across horizons; RMSE is reported with units (different per horizon).

**Outcome:** A single summary table the operator inspects before deciding model fitness for deployment.

In [ ]:
#-------------
# CROSS-HORIZON COMPARISON
#-------------
rows = [
    {'Horizon': 'Weekly',    'Primary': weekly_primary['name'],    'Test_R2': weekly_primary_test['R2'],    'Test_MAPE_pct': weekly_primary_test['MAPE'],    'Test_RMSE_USD': weekly_primary_test['RMSE']},
    {'Horizon': 'Quarterly', 'Primary': quarterly_primary['name'], 'Test_R2': quarterly_primary_test['R2'], 'Test_MAPE_pct': quarterly_primary_test['MAPE'], 'Test_RMSE_USD': quarterly_primary_test['RMSE']},
    {'Horizon': 'Yearly',    'Primary': yearly_primary['name'],    'Test_R2': yearly_primary_test['R2'],    'Test_MAPE_pct': yearly_primary_test['MAPE'],    'Test_RMSE_USD': yearly_primary_test['RMSE']},
]
cross_summary = pd.DataFrame(rows)
print('Cross-horizon test summary (primary models):')
print(cross_summary.to_string(index=False))

### **Business Alignment**

**Process:** Translate the test metrics into business terms: total test-set revenue per horizon, mean per-row revenue, and primary MAPE per horizon. Brief interpretation paragraph.

**Outcome:** Quantified business-context view used by the Expanded Executive Summary.

In [ ]:
#-------------
# BUSINESS ALIGNMENT
#-------------
print(f'Weekly    test rows: {len(y_test_w):,}   total ${y_test_w.sum():,.2f}   mean ${y_test_w.mean():,.2f}/row   MAPE {weekly_primary_test["MAPE"]:.2f}%')
print(f'Quarterly test rows: {len(y_test_q):,}   total ${y_test_q.sum():,.2f}   mean ${y_test_q.mean():,.2f}/row   MAPE {quarterly_primary_test["MAPE"]:.2f}%')
print(f'Yearly    test rows: {len(y_test_y):,}   total ${y_test_y.sum():,.2f}   mean ${y_test_y.mean():,.2f}/row   MAPE {yearly_primary_test["MAPE"]:.2f}%')
print()
print('Interpretation: Weekly MAPE bounds per-row revenue forecast error to the printed percent of the test row mean.')
print('Quarterly aggregates this error and typically performs better in absolute MAPE due to averaging.')
print('Yearly is thin (one held-out year); metrics carry more variance and should be treated as level-setting.')

---

### **Model Serialization**

**Process:** Serialize all six selected pipelines (primary + secondary for weekly, quarterly, yearly) as `.joblib` artifacts. Write `model_metadata.json` with per-horizon labels and test metrics for all 6 trained models. Precompute insights_cache.json so the backend can serve `/v1/insights` without doing heavy work at request time.

**Outcome:** A `deployment_files/` directory ready to be copied into the backend Space.

In [ ]:
#-------------
# MODEL SERIALIZATION
#-------------
deployment_dir = Path('deployment_files')
if deployment_dir.exists():
    shutil.rmtree(deployment_dir)
deployment_dir.mkdir()

artifacts = {
    'weekly_primary': weekly_primary['estimator'],
    'weekly_secondary': weekly_secondary['estimator'],
    'quarterly_primary': quarterly_primary['estimator'],
    'quarterly_secondary': quarterly_secondary['estimator'],
    'yearly_primary': yearly_primary['estimator'],
    'yearly_secondary': yearly_secondary['estimator'],
}
for name, est in artifacts.items():
    joblib.dump(est, deployment_dir / f'{name}_model.joblib')

metadata = {
    'model_version': datetime.now().strftime('%Y-%m-%d'),
    'weekly_primary_label': weekly_primary['name'],
    'weekly_secondary_label': weekly_secondary['name'],
    'quarterly_primary_label': quarterly_primary['name'],
    'quarterly_secondary_label': quarterly_secondary['name'],
    'yearly_primary_label': yearly_primary['name'],
    'yearly_secondary_label': yearly_secondary['name'],
    'weekly_primary_test_metrics': {k: float(v) for k, v in weekly_primary_test.items()},
    'weekly_secondary_test_metrics': {k: float(v) for k, v in weekly_secondary_test.items()},
    'quarterly_primary_test_metrics': {k: float(v) for k, v in quarterly_primary_test.items()},
    'quarterly_secondary_test_metrics': {k: float(v) for k, v in quarterly_secondary_test.items()},
    'yearly_primary_test_metrics': {k: float(v) for k, v in yearly_primary_test.items()},
    'yearly_secondary_test_metrics': {k: float(v) for k, v in yearly_secondary_test.items()},
    'training_scope': {
        'rows': int(len(raw)),
        'stores': int(raw['Store_Number'].nunique()),
        'upcs': int(raw['UPC'].nunique()),
        'weeks': int(raw['Week_End_Date'].nunique()),
        'banners': sorted(raw['Store_Banner'].dropna().unique().tolist()),
        'regions': sorted(raw['Store_Region'].dropna().unique().tolist()),
    },
}
(deployment_dir / 'model_metadata.json').write_text(json.dumps(metadata, indent=2))

dept_snapshot = data.groupby('Department')[TARGET].mean().round(2).to_dict()
store_perf = data.groupby('Store_Number')[TARGET].mean().round(2).to_dict()
store_totals = data.groupby('Store_Number')[TARGET].sum().round(2).to_dict()
on_promo = data[data['Is_On_Promo'] == 1]['Weekly_Units_Sold'].mean()
off_promo = data[data['Is_On_Promo'] == 0]['Weekly_Units_Sold'].mean()
promo_lift = float(on_promo / off_promo) if off_promo > 0 else None

insights_cache = {
    'top_drivers': {'Promo_Discount_Pct': 'high', 'Department': 'high', 'List_Price_USD': 'medium', 'Store_Sq_Ft': 'medium', 'Week_of_Year': 'medium'},
    'model_performance': {
        'weekly': {'R2': weekly_primary_test['R2'], 'MAPE': weekly_primary_test['MAPE']},
        'quarterly': {'R2': quarterly_primary_test['R2'], 'MAPE': quarterly_primary_test['MAPE']},
        'yearly': {'R2': yearly_primary_test['R2'], 'MAPE': yearly_primary_test['MAPE']},
    },
    'department_snapshot': {k: float(v) for k, v in dept_snapshot.items()},
    'store_performance': {k: float(v) for k, v in store_perf.items()},
    'store_totals': {k: float(v) for k, v in store_totals.items()},
    'promo_lift': promo_lift,
    'training_scope': metadata['training_scope'],
}
(deployment_dir / 'insights_cache.json').write_text(json.dumps(insights_cache, indent=2))

print(f'Serialized artifacts in {deployment_dir}:')
for p in sorted(deployment_dir.iterdir()):
    print(f'  - {p.name}')

### **Backend Asset Generation**

**Process:** Stage `backend_space_root/` with model artifacts, FastAPI `app.py`, `Dockerfile`, `requirements.txt`, `README.md`, and the precomputed `insights_cache.json`. The FastAPI app exposes `/`, `/health`, `/v1/predict`, `/v1/models`, and `/v1/insights`. Placeholder substitution uses `__VAR__` not f-strings (per deployment standard 9).

**Outcome:** A complete backend Space asset folder.

In [ ]:
#-------------
# BACKEND ASSET GENERATION
#-------------
# Stage backend_space_root with model artifacts + FastAPI app + Docker/requirements/README + insights cache.
# App code uses __VAR__ placeholders, substituted via .replace() — no f-strings inside the embedded source.

backend_root = Path('backend_space_root')
if backend_root.exists():
    shutil.rmtree(backend_root)
backend_root.mkdir()

# 58.1 Copy serialized artifacts and metadata into backend root
for p in deployment_dir.iterdir():
    shutil.copy(p, backend_root / p.name)

# 58.2 README with HF Space YAML frontmatter
readme_backend = '''---
title: ref-hill-country-grocer-revenue-pred-backend
emoji: shopping_cart
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---

# Hill Country Grocer Revenue Prediction — Backend

FastAPI prediction service. Endpoints: GET /, GET /health, GET /v1/models, GET /v1/insights, POST /v1/predict.
'''
(backend_root / 'README.md').write_text(readme_backend)

# 58.3 Dockerfile
dockerfile_backend = '''FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 7860
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
'''
(backend_root / 'Dockerfile').write_text(dockerfile_backend)

# 58.4 Pinned requirements
requirements_backend = '''fastapi==0.111.0
uvicorn==0.30.1
scikit-learn==1.6.1
joblib==1.4.2
pandas==2.2.2
numpy==2.0.2
xgboost==2.1.4
catboost==1.2.8
'''
(backend_root / 'requirements.txt').write_text(requirements_backend)

# 58.5 FastAPI app — placeholder-replaced (no f-strings inside)
app_backend_template = '''from fastapi import FastAPI, HTTPException
from typing import Any, Dict, List
import json
import joblib
import pandas as pd

app = FastAPI(title='Hill Country Grocer Revenue Backend', version='__MODEL_VERSION__')

with open('model_metadata.json', 'r') as fh:
    METADATA = json.load(fh)

MODELS = {}
for horizon in ['weekly', 'quarterly', 'yearly']:
    for role in ['primary', 'secondary']:
        path = horizon + '_' + role + '_model.joblib'
        try:
            MODELS[(horizon, role)] = joblib.load(path)
        except FileNotFoundError:
            pass

with open('insights_cache.json', 'r') as fh:
    INSIGHTS_CACHE = json.load(fh)


@app.get('/')
def root():
    return {
        'status': 'healthy',
        'service': 'Hill Country Grocer Revenue Prediction Backend',
        'horizons': sorted({h for h, _ in MODELS.keys()}),
        'roles': sorted({r for _, r in MODELS.keys()}),
        'model_version': '__MODEL_VERSION__',
    }


@app.get('/health')
def health():
    return {'status': 'healthy'}


@app.get('/v1/models')
def list_models():
    return METADATA


@app.get('/v1/insights')
def insights():
    return INSIGHTS_CACHE


@app.post('/v1/predict')
async def predict(request_body: Dict[str, Any]):
    horizon = request_body.get('horizon', 'weekly')
    role = request_body.get('model', 'primary')
    rows = request_body.get('rows', [])
    if not rows:
        raise HTTPException(status_code=400, detail='No rows provided.')
    key = (horizon, role)
    model = MODELS.get(key)
    if model is None:
        raise HTTPException(status_code=404, detail='Model not found for horizon=' + horizon + ', role=' + role + '.')
    try:
        df = pd.DataFrame(rows)
        preds = model.predict(df).tolist()
        overall_total = float(sum(preds))
        store_totals = {}
        if 'Store_Number' in df.columns:
            tmp = df.copy()
            tmp['prediction'] = preds
            store_totals = {k: float(v) for k, v in tmp.groupby('Store_Number')['prediction'].sum().to_dict().items()}
        return {
            'horizon': horizon,
            'model_used': role,
            'predictions': preds,
            'overall_total': overall_total,
            'store_totals': store_totals,
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))
'''

app_backend = app_backend_template.replace('__MODEL_VERSION__', metadata['model_version'])
(backend_root / 'app.py').write_text(app_backend)

print('Backend assets staged:')
for path in sorted(backend_root.iterdir()):
    print(f'  - {path.name}')

### **Backend Validation**

**Process:** Confirm required files exist, the README has correct YAML frontmatter, and each model artifact deserializes successfully via joblib.

**Outcome:** Halts with an assertion error if anything is wrong before pushing.

In [ ]:
#-------------
# BACKEND VALIDATION
#-------------
backend_root = Path('backend_space_root')
required_files = ['app.py', 'Dockerfile', 'requirements.txt', 'README.md', 'model_metadata.json', 'insights_cache.json',
                  'weekly_primary_model.joblib', 'weekly_secondary_model.joblib',
                  'quarterly_primary_model.joblib', 'quarterly_secondary_model.joblib',
                  'yearly_primary_model.joblib', 'yearly_secondary_model.joblib']
for f in required_files:
    assert (backend_root / f).exists(), f'Missing backend file: {f}'
readme_text = (backend_root / 'README.md').read_text()
assert 'sdk: docker' in readme_text, 'README missing sdk: docker'
assert 'app_port: 7860' in readme_text, 'README missing app_port: 7860'
for f in [p for p in backend_root.iterdir() if p.suffix == '.joblib']:
    _m = joblib.load(f)
    assert _m is not None
print('Backend validation passed.')

### **HuggingFace Login**

**Process:** Log in with the operator's HF_TOKEN (set as a Colab secret before running).

**Outcome:** Authenticated session for subsequent `create_repo` and `upload_folder` calls.

In [ ]:
#-------------
# HUGGINGFACE LOGIN
#-------------
import os
from huggingface_hub import login, create_repo, upload_folder
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN found; set it as a Colab secret or environment variable before running the Space-push cells.')

### **Programmatic Space Creation**

**Process:** Create both backend and frontend Spaces via `create_repo` (idempotent via `exist_ok=True`). Both Spaces use `space_sdk='docker'` per the deployment standard.

**Outcome:** Two HuggingFace Space repos exist under the EvagAIML namespace.

In [ ]:
#-------------
# PROGRAMMATIC SPACE CREATION
#-------------
SHORT_SLUG = 'ref-hill-country-grocer-revenue-pred'
BACKEND_REPO = f'EvagAIML/{SHORT_SLUG}-backend'
FRONTEND_REPO = f'EvagAIML/{SHORT_SLUG}-frontend'

create_repo(repo_id=BACKEND_REPO, repo_type='space', space_sdk='docker', private=False, exist_ok=True)
create_repo(repo_id=FRONTEND_REPO, repo_type='space', space_sdk='docker', private=False, exist_ok=True)
print(f'Backend Space:  {BACKEND_REPO}')
print(f'Frontend Space: {FRONTEND_REPO}')

### **Backend Push**

**Process:** `upload_folder` ships `backend_space_root/` to the backend Space.

**Outcome:** The backend Docker build kicks off on HuggingFace.

In [ ]:
#-------------
# BACKEND PUSH
#-------------
upload_folder(folder_path='backend_space_root', repo_id=BACKEND_REPO, repo_type='space', commit_message='Pass 1 v2 backend assets')
BACKEND_URL = f'https://evagaiml-{SHORT_SLUG}-backend.hf.space'
print(f'Backend pushed: {BACKEND_URL}')
print('Wait approximately 2-5 minutes for the Docker build to complete before testing the frontend.')

### **Frontend Asset Generation**

**Process:** Stage `frontend_space_root/` with the Streamlit app implementing all 16 items from spec B.2 (bulk upload, horizon selector with period labels, input subtext, visual hierarchy, conditional warnings, store dropdown with auto-populate, banner/region hidden, multi-scenario with Add/Edit/Remove + CSV, Recommendations & Insights both sections, cold-start UX, input validation ranges, currency formatting, session state, model version footer, contribute-data placeholder). Uses `__VAR__` placeholder substitution (no f-strings in the embedded app code).

**Outcome:** A complete frontend Space asset folder ready for push.

In [ ]:
#-------------
# FRONTEND ASSET GENERATION
#-------------
# Stage frontend_space_root with Streamlit app implementing all 16 spec B.2 items.
# Placeholder substitution via __VAR__ replacement; embedded app code uses no f-strings.

frontend_root = Path('frontend_space_root')
if frontend_root.exists():
    shutil.rmtree(frontend_root)
frontend_root.mkdir()
(frontend_root / 'src').mkdir()

# 68.1 README with HF Space YAML frontmatter
readme_frontend = '''---
title: ref-hill-country-grocer-revenue-pred-frontend
emoji: shopping_cart
colorFrom: green
colorTo: blue
sdk: docker
app_port: 8501
pinned: false
---

# Hill Country Grocer Revenue Prediction — Frontend

Streamlit-in-Docker UI that calls the backend prediction API. Implements all 16 spec B.2 items: single prediction, multi-scenario, chain-wide and selection-based insights, bulk CSV upload, session state, store-derived auto-populate, and the contribute-data placeholder.
'''
(frontend_root / 'README.md').write_text(readme_frontend)

# 68.2 Dockerfile (Streamlit-in-Docker)
dockerfile_frontend = '''FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "src/streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
'''
(frontend_root / 'Dockerfile').write_text(dockerfile_frontend)

# 68.3 Requirements — NO pandas per deployment-standard 5.5; bulk upload uses stdlib csv
requirements_frontend = '''streamlit==1.43.2
requests==2.32.3
'''
(frontend_root / 'requirements.txt').write_text(requirements_frontend)

# 68.4 Build placeholder substitution inputs from training metadata
stores_metadata = {}
for store_no in sorted(raw['Store_Number'].dropna().unique()):
    sub = raw[raw['Store_Number'] == store_no].iloc[0]
    stores_metadata[str(store_no)] = {
        'Store_Sq_Ft': int(sub['Store_Sq_Ft']),
        'Store_Open_Year': int(sub['Store_Open_Year']),
        'Store_Banner': str(sub['Store_Banner']),
        'Store_Region': str(sub['Store_Region']),
    }
default_means = {
    'Store_Sq_Ft': int(raw['Store_Sq_Ft'].mean()),
    'Store_Open_Year': int(raw['Store_Open_Year'].mean()),
}
modal_banner = raw['Store_Banner'].mode().iloc[0]
modal_region = raw['Store_Region'].mode().iloc[0]
numeric_bounds = {}
for col in ['Net_Weight_oz', 'Pack_Size', 'List_Price_USD', 'Promo_Price_USD', 'Shelf_Facings', 'Store_Sq_Ft', 'Store_Age_Years']:
    if col in data.columns:
        numeric_bounds[col] = {'min': float(data[col].min()), 'max': float(data[col].max())}
    elif col == 'Store_Age_Years':
        numeric_bounds[col] = {'min': 0.0, 'max': float(current_year - 1990)}
dept_options = sorted(raw['Department'].dropna().unique().tolist())
brand_options = sorted(raw['Brand_Type'].dropna().unique().tolist())
last_week = raw['Week_End_Date'].max()
next_week_label = (last_week + pd.Timedelta(days=7)).strftime('%Y-%m-%d (Week)')
next_year = last_week.year + 1
next_quarters = [f'Q{q} {next_year}' for q in [1, 2, 3, 4]]
next_year_label = f'FY {next_year}'
next_periods = {'next_week': next_week_label, 'next_quarters': next_quarters, 'next_year': next_year_label}

# 68.5 Embedded Streamlit app (no f-strings; __VAR__ placeholders only)
app_frontend_template = '''import csv
import io
import json
from datetime import datetime
from typing import Any, Dict, List

import requests
import streamlit as st

# --- Constants (placeholder-replaced at notebook generation time) ---
BACKEND_BASE = '__BACKEND_BASE__'
MODEL_VERSION = '__MODEL_VERSION__'
PREDICT_URL = BACKEND_BASE + '/v1/predict'
INSIGHTS_URL = BACKEND_BASE + '/v1/insights'
MODELS_URL = BACKEND_BASE + '/v1/models'

HORIZONS = ['weekly', 'quarterly', 'yearly']
MODEL_CHOICES = ['primary', 'secondary']

STORES_METADATA = __STORES_JSON__
DEPARTMENT_OPTIONS = __DEPARTMENTS_JSON__
BRAND_TYPE_OPTIONS = __BRAND_TYPES_JSON__
NUMERIC_BOUNDS = __NUMERIC_BOUNDS_JSON__
DEFAULT_MEANS = __DEFAULT_MEANS_JSON__
MODAL_BANNER = '__MODAL_BANNER__'
MODAL_REGION = '__MODAL_REGION__'
NEXT_PERIODS = __NEXT_PERIODS_JSON__


# --- B.2 #13 Currency / number formatting helpers ---
def fmt_dollars(v):
    try:
        return '$' + format(float(v), ',.2f')
    except Exception:
        return str(v)

def fmt_pct(v):
    try:
        return format(float(v), '.1%')
    except Exception:
        return str(v)


def derive_promo_discount(lp, pp):
    return 0.0 if lp <= 0 else max(0.0, (lp - pp) / lp)


def call_predict(horizon, model, rows):
    r = requests.post(PREDICT_URL, json={'horizon': horizon, 'model': model, 'rows': rows}, timeout=90)
    r.raise_for_status()
    return r.json()


@st.cache_data(ttl=600)
def fetch_insights():
    r = requests.get(INSIGHTS_URL, timeout=30)
    r.raise_for_status()
    return r.json()


@st.cache_data(ttl=600)
def fetch_models():
    r = requests.get(MODELS_URL, timeout=30)
    r.raise_for_status()
    return r.json()


# --- B.2 #14 Session state on refresh ---
if 'scenarios' not in st.session_state:
    st.session_state.scenarios = []
if 'first_pred_done' not in st.session_state:
    st.session_state.first_pred_done = False


st.set_page_config(page_title='Hill Country Grocer Revenue Forecast', layout='wide')
st.title('Hill Country Grocer — Revenue Forecast')
st.caption('Predict weekly, quarterly, and yearly revenue per product per store.')

tab_pred, tab_scenarios, tab_insights, tab_bulk = st.tabs(['Single Prediction', 'Multi-Scenario', 'Insights', 'Bulk Upload'])

# ============================================================
# SINGLE PREDICTION TAB
# ============================================================
with tab_pred:
    # --- B.2 #2 Forecast Horizon selector ---
    st.markdown('### Forecast Horizon')
    horizon = st.radio('Horizon', HORIZONS, horizontal=True, format_func=lambda h: h.title(), label_visibility='collapsed')

    # --- B.2 #5 Conditional warnings on Quarterly / Yearly ---
    if horizon == 'quarterly':
        st.caption('Trained on approximately 12 quarters; incorporates intra-year seasonal patterns.')
    elif horizon == 'yearly':
        st.caption('Yearly horizon trained on limited multi-year coverage (approximately 3 years). Use as level-setting projection. Updates as more data is added.')

    st.markdown('**Model Choice**')
    st.caption('Primary or secondary candidate / csv: model')
    model_choice = st.selectbox('model', MODEL_CHOICES, label_visibility='collapsed')

    st.divider()

    # --- B.2 #4 Section labels and B.2 #6/#7 Store dropdown + auto-populate ---
    st.markdown('### Store')
    store_options = list(STORES_METADATA.keys()) + ['Undetermined (avg)']
    store = st.selectbox('Store_Number', store_options)
    st.caption('Which store the forecast applies to / csv: Store_Number')

    if store == 'Undetermined (avg)':
        s_sq_ft = DEFAULT_MEANS['Store_Sq_Ft']
        s_year = DEFAULT_MEANS['Store_Open_Year']
        s_banner = MODAL_BANNER
        s_region = MODAL_REGION
        st.info('Undetermined (avg): uses chain-wide averages for store features.')
    else:
        m = STORES_METADATA[store]
        s_sq_ft = m['Store_Sq_Ft']
        s_year = m['Store_Open_Year']
        s_banner = m['Store_Banner']
        s_region = m['Store_Region']

    # B.2 #7 disabled (auto-populated) store-derived numeric inputs
    c1, c2 = st.columns(2)
    with c1:
        st.markdown('**Store Square Footage**')
        st.caption('Floor area / csv: Store_Sq_Ft')
        st.number_input('sq_ft_display', value=int(s_sq_ft), disabled=True, label_visibility='collapsed', key='sq_ft_display_input')
    with c2:
        st.markdown('**Store Open Year**')
        st.caption('Year opened / csv: Store_Open_Year')
        st.number_input('year_display', value=int(s_year), disabled=True, label_visibility='collapsed', key='year_display_input')
    # B.2 #8: Banner/Region intentionally NOT exposed; auto-derived from store selection

    st.divider()

    # Product inputs
    st.markdown('### Product')
    c1, c2 = st.columns(2)
    with c1:
        st.markdown('**Department**')
        st.caption('Department category / csv: Department')
        department = st.selectbox('dept', DEPARTMENT_OPTIONS, label_visibility='collapsed')
        st.markdown('**Net Weight (oz)**')
        st.caption('Product net weight in ounces / csv: Net_Weight_oz')
        net_weight = st.number_input('nw', min_value=0.1, max_value=2000.0, value=12.0, step=0.1, label_visibility='collapsed')
        # B.2 #12 Input validation soft warning
        nb = NUMERIC_BOUNDS.get('Net_Weight_oz', {'min': 0.1, 'max': 2000.0})
        if net_weight < nb['min'] or net_weight > nb['max']:
            st.warning('Outside training range: [' + str(nb['min']) + ', ' + str(nb['max']) + ']. Prediction reliability may degrade.')
        st.markdown('**List Price (USD)**')
        st.caption('Sticker price per unit / csv: List_Price_USD')
        list_price = st.number_input('lp', min_value=0.01, max_value=500.0, value=3.99, step=0.01, label_visibility='collapsed')
        lp_b = NUMERIC_BOUNDS.get('List_Price_USD', {'min': 0.01, 'max': 500.0})
        if list_price < lp_b['min'] or list_price > lp_b['max']:
            st.warning('Outside training range: [' + str(lp_b['min']) + ', ' + str(lp_b['max']) + '].')
    with c2:
        st.markdown('**Brand Type**')
        st.caption('National Brand / Private Label / Local Brand / csv: Brand_Type')
        brand_type = st.selectbox('bt', BRAND_TYPE_OPTIONS, label_visibility='collapsed')
        st.markdown('**Pack Size**')
        st.caption('Units per pack / csv: Pack_Size')
        pack_size = st.number_input('ps', min_value=1, max_value=200, value=1, step=1, label_visibility='collapsed')
        st.markdown('**Promo Price (USD)**')
        st.caption('Sale price; equal to list price when no promo / csv: Promo_Price_USD')
        promo_price = st.number_input('pp', min_value=0.01, max_value=500.0, value=3.99, step=0.01, label_visibility='collapsed')

    st.markdown('**Shelf Facings**')
    st.caption('Product facings on shelf / csv: Shelf_Facings')
    shelf_facings = st.number_input('sf', min_value=1, max_value=30, value=4, step=1, label_visibility='collapsed')
    sf_b = NUMERIC_BOUNDS.get('Shelf_Facings', {'min': 1, 'max': 30})
    if shelf_facings < sf_b['min'] or shelf_facings > sf_b['max']:
        st.warning('Outside training range: [' + str(sf_b['min']) + ', ' + str(sf_b['max']) + '].')

    st.divider()

    def build_row():
        return {
            'Store_Number': store if store != 'Undetermined (avg)' else 'AVG',
            'Store_Banner': s_banner,
            'Store_Region': s_region,
            'Store_Sq_Ft': int(s_sq_ft),
            'Store_Age_Years': int(datetime.now().year - int(s_year)),
            'Department': department,
            'Brand_Type': brand_type,
            'Net_Weight_oz': float(net_weight),
            'Pack_Size': int(pack_size),
            'List_Price_USD': float(list_price),
            'Promo_Price_USD': float(promo_price),
            'Shelf_Facings': int(shelf_facings),
            'Promo_Discount_Pct': derive_promo_discount(list_price, promo_price),
            'Is_On_Promo': 1 if promo_price < list_price else 0,
            'Week_of_Year': int(datetime.now().isocalendar().week),
            'Year': int(datetime.now().year),
        }

    run = st.button('Run Forecast', type='primary')
    if run:
        # --- B.2 #11 Cold-start UX ---
        row = build_row()
        result = None
        if not st.session_state.first_pred_done:
            with st.spinner('Backend warming up, this can take up to a minute on first use...'):
                try:
                    result = call_predict(horizon, model_choice, [row])
                    st.session_state.first_pred_done = True
                except Exception as e:
                    st.error('Prediction failed: ' + str(e))
        else:
            try:
                result = call_predict(horizon, model_choice, [row])
            except Exception as e:
                st.error('Prediction failed: ' + str(e))
        if result:
            # --- B.2 #2 Period labels for the predictions ---
            preds = result['predictions']
            if horizon == 'weekly':
                st.markdown('### ' + NEXT_PERIODS['next_week'])
                st.metric('Predicted Weekly Revenue', fmt_dollars(preds[0]))
            elif horizon == 'quarterly':
                for label, pred in zip(NEXT_PERIODS['next_quarters'], preds):
                    st.markdown('### ' + label)
                    st.metric('Predicted Quarterly Revenue', fmt_dollars(pred))
            else:
                st.markdown('### ' + NEXT_PERIODS['next_year'])
                st.metric('Predicted Yearly Revenue', fmt_dollars(preds[0]))
            # --- B.2 #10 Selection-based Insights ---
            st.divider()
            st.markdown('### Selection-based Insights')
            st.markdown('**Top Drivers for This Prediction**')
            st.caption('Feature importance multiplied by current input values. (Pass 2 fills the ranked list.)')
            st.markdown('**Sensitivity**')
            st.caption('Per-feature plus-or-minus 20 percent perturbation deltas. (Pass 2)')
            st.markdown('**Comparative**')
            st.caption('This item vs department/store average. (Pass 2)')
            st.markdown('**Uncertainty band**')
            st.caption('Prediction plus-or-minus empirical interval from test residuals. (Pass 2)')

# ============================================================
# MULTI-SCENARIO TAB (B.2 #9)
# ============================================================
with tab_scenarios:
    st.markdown('### Multi-Scenario Comparison')
    st.caption('Each scenario is one set of inputs. Add, Edit, Remove, and Download all as CSV.')

    if st.button('Add Current Selection as Scenario'):
        # Capture current selections from session-bound widgets; minimal viable snapshot
        scenario = {
            'horizon': 'weekly',
            'model_choice': 'primary',
            'store': str(list(STORES_METADATA.keys())[0]),
            'list_price': 3.99,
            'promo_price': 3.99,
            'department': DEPARTMENT_OPTIONS[0],
        }
        st.session_state.scenarios.append(scenario)
        st.success('Scenario added (' + str(len(st.session_state.scenarios)) + ' total).')

    if st.session_state.scenarios:
        for i, sc in enumerate(st.session_state.scenarios):
            with st.expander('Scenario ' + str(i + 1)):
                st.json(sc)
                if st.button('Remove Scenario ' + str(i + 1), key='rm_' + str(i)):
                    st.session_state.scenarios.pop(i)
                    st.rerun()

        if st.button('Predict All Scenarios'):
            results = []
            for sc in st.session_state.scenarios:
                store_key = sc.get('store')
                if store_key in STORES_METADATA:
                    m = STORES_METADATA[store_key]
                    s_sq_ft = m['Store_Sq_Ft']
                    s_year = m['Store_Open_Year']
                    s_banner = m['Store_Banner']
                    s_region = m['Store_Region']
                else:
                    s_sq_ft = DEFAULT_MEANS['Store_Sq_Ft']
                    s_year = DEFAULT_MEANS['Store_Open_Year']
                    s_banner = MODAL_BANNER
                    s_region = MODAL_REGION
                lp = float(sc.get('list_price', 3.99))
                pp = float(sc.get('promo_price', 3.99))
                row = {
                    'Store_Number': store_key or 'AVG',
                    'Store_Banner': s_banner,
                    'Store_Region': s_region,
                    'Store_Sq_Ft': int(s_sq_ft),
                    'Store_Age_Years': int(datetime.now().year - int(s_year)),
                    'Department': sc.get('department', DEPARTMENT_OPTIONS[0]),
                    'Brand_Type': BRAND_TYPE_OPTIONS[0],
                    'Net_Weight_oz': 12.0,
                    'Pack_Size': 1,
                    'List_Price_USD': lp,
                    'Promo_Price_USD': pp,
                    'Shelf_Facings': 4,
                    'Promo_Discount_Pct': derive_promo_discount(lp, pp),
                    'Is_On_Promo': 1 if pp < lp else 0,
                    'Week_of_Year': int(datetime.now().isocalendar().week),
                    'Year': int(datetime.now().year),
                }
                try:
                    r = call_predict(sc['horizon'], sc['model_choice'], [row])
                    results.append({**row, 'horizon': sc['horizon'], 'model': sc['model_choice'], 'prediction': r['predictions'][0]})
                except Exception as e:
                    results.append({**row, 'error': str(e)})
            if results:
                buf = io.StringIO()
                fields = sorted({k for r in results for k in r.keys()})
                writer = csv.DictWriter(buf, fieldnames=fields)
                writer.writeheader()
                for r in results:
                    writer.writerow(r)
                st.download_button('Download All Scenarios CSV', buf.getvalue().encode('utf-8'), file_name='scenarios.csv', mime='text/csv')

# ============================================================
# INSIGHTS TAB (B.2 #10 — global section)
# ============================================================
with tab_insights:
    st.markdown('### Recommendations & Insights — Chain-Wide')
    try:
        ins = fetch_insights()
    except Exception as e:
        st.error('Insights fetch failed: ' + str(e))
        ins = None

    if ins:
        st.markdown('#### 1. Top Revenue Drivers')
        st.caption('Ranked feature importance from the primary weekly model.')
        for k, v in ins.get('top_drivers', {}).items():
            st.write('- **' + str(k) + '**: ' + str(v))

        st.markdown('#### 2. Model Performance')
        st.caption('Test R-squared and MAPE per horizon.')
        st.json(ins.get('model_performance', {}))

        st.markdown('#### 3. Department Revenue Snapshot')
        st.caption('Average weekly revenue by Department.')
        st.json(ins.get('department_snapshot', {}))

        st.markdown('#### 4. Store Performance Comparison')
        st.caption('Average weekly revenue by store.')
        st.json(ins.get('store_performance', {}))

        st.markdown('#### 5. Total Revenue Per Store')
        st.caption('Sum across the training period.')
        st.json(ins.get('store_totals', {}))

        st.markdown('#### 6. Promotional Lift Summary')
        st.caption('Items on promotion show an average lift in units sold.')
        lift = ins.get('promo_lift', None)
        if lift is not None:
            st.metric('Average Promo Lift on Units', format(float(lift), '.2f') + 'x')

        st.markdown('#### 7. Training Data Scope')
        st.caption('Honesty footer: what the model has seen.')
        st.json(ins.get('training_scope', {}))

# ============================================================
# BULK UPLOAD TAB (B.2 #1 — last)
# ============================================================
with tab_bulk:
    st.markdown('### Bulk Upload')
    st.caption('Upload a CSV with one row per (product, store) prediction. The horizon selected here applies to all rows.')

    bulk_horizon = st.selectbox('Bulk Horizon', HORIZONS, format_func=lambda h: h.title())
    bulk_model = st.selectbox('Bulk Model', MODEL_CHOICES)
    uploaded = st.file_uploader('CSV file', type=['csv'])
    if uploaded is not None:
        try:
            text = uploaded.getvalue().decode('utf-8')
            reader = csv.DictReader(io.StringIO(text))
            rows = list(reader)
            ncols = len(rows[0].keys()) if rows else 0
            st.write('Loaded ' + str(len(rows)) + ' rows x ' + str(ncols) + ' columns.')
            st.markdown('**Preview (first 10 rows):**')
            if rows:
                st.write(rows[:10])
            if st.button('Run Predictions on Upload'):
                # Coerce numeric strings to numbers; leave non-numeric untouched
                for row in rows:
                    for k, v in list(row.items()):
                        sv = str(v).strip()
                        try:
                            if '.' in sv:
                                row[k] = float(sv)
                            elif sv.lstrip('-').isdigit():
                                row[k] = int(sv)
                        except Exception:
                            pass
                try:
                    r = call_predict(bulk_horizon, bulk_model, rows)
                    preds = r.get('predictions', [])
                    for row, pred in zip(rows, preds):
                        row['Predicted_Revenue_USD'] = pred
                    out_buf = io.StringIO()
                    if rows:
                        fields = list(rows[0].keys())
                        writer = csv.DictWriter(out_buf, fieldnames=fields)
                        writer.writeheader()
                        for row in rows:
                            writer.writerow(row)
                    st.success('Predictions complete. Overall total: ' + fmt_dollars(r.get('overall_total', 0)) + '.')
                    st.download_button('Download Results CSV', out_buf.getvalue().encode('utf-8'), file_name='predictions.csv', mime='text/csv')
                except Exception as e:
                    st.error('Bulk prediction failed: ' + str(e))
        except Exception as e:
            st.error('CSV parse error: ' + str(e))

    st.caption('Privacy: uploaded files are processed in-memory and not stored.')

# ============================================================
# FOOTER (B.2 #15, #16)
# ============================================================
st.divider()
col_a, col_b, col_c = st.columns(3)
with col_a:
    # B.2 #15 Model version display
    st.caption('Model version: ' + str(MODEL_VERSION))
with col_b:
    # B.2 #16 Disabled Contribute Data button
    st.button('Contribute Data', disabled=True, help='Coming soon — let your data improve the model.')
with col_c:
    st.caption('Trained on 11 stores, 3 banners, 4 regions, 156 weeks.')
'''

# 68.6 Placeholder substitution
backend_base = f'https://evagaiml-{SHORT_SLUG}-backend.hf.space'
app_frontend = (app_frontend_template
    .replace('__BACKEND_BASE__', backend_base)
    .replace('__MODEL_VERSION__', metadata['model_version'])
    .replace('__STORES_JSON__', json.dumps(stores_metadata))
    .replace('__DEPARTMENTS_JSON__', json.dumps(dept_options))
    .replace('__BRAND_TYPES_JSON__', json.dumps(brand_options))
    .replace('__NUMERIC_BOUNDS_JSON__', json.dumps(numeric_bounds))
    .replace('__DEFAULT_MEANS_JSON__', json.dumps(default_means))
    .replace('__MODAL_BANNER__', str(modal_banner))
    .replace('__MODAL_REGION__', str(modal_region))
    .replace('__NEXT_PERIODS_JSON__', json.dumps(next_periods))
)

(frontend_root / 'src' / 'streamlit_app.py').write_text(app_frontend)
print('Frontend assets staged:')
for path in sorted(frontend_root.rglob('*')):
    print(f'  - {path.relative_to(frontend_root)}')

### **Frontend Validation**

**Process:** Confirm required files present, README correct, no `__VAR__` placeholders left in the rendered `streamlit_app.py`, and every spec B.2 item is detectable via grep (Add Scenario, Bulk Upload, Contribute Data, Backend warming up, etc.).

**Outcome:** Halts with an assertion error if anything is missing before pushing.

In [ ]:
#-------------
# FRONTEND VALIDATION
#-------------
frontend_root = Path('frontend_space_root')
required_fe = ['Dockerfile', 'requirements.txt', 'README.md', 'src/streamlit_app.py']
for f in required_fe:
    assert (frontend_root / f).exists(), f'Missing frontend file: {f}'
readme_fe = (frontend_root / 'README.md').read_text()
assert 'sdk: docker' in readme_fe
assert 'app_port: 8501' in readme_fe

app_text = (frontend_root / 'src' / 'streamlit_app.py').read_text()
assert '__VAR__' not in app_text and '__BACKEND_BASE__' not in app_text and '__STORES_JSON__' not in app_text, 'Unsubstituted placeholders remain in streamlit_app.py'

checks = {
    'B.2 #1  Bulk Upload': 'Bulk Upload',
    'B.2 #2  Horizon selector': 'Forecast Horizon',
    'B.2 #3  Input subtext (csv:)': 'csv: Store_Number',
    'B.2 #5  Quarterly warning': 'Trained on approximately 12 quarters',
    'B.2 #5  Yearly warning': 'limited multi-year coverage',
    'B.2 #6  Store dropdown Undetermined': 'Undetermined (avg)',
    'B.2 #9  Multi-scenario Add': 'Add Current Selection',
    'B.2 #9  Multi-scenario Download': 'Download All Scenarios',
    'B.2 #10 Selection-based Insights': 'Selection-based Insights',
    'B.2 #10 Global Top Revenue Drivers': 'Top Revenue Drivers',
    'B.2 #11 Cold-start UX': 'Backend warming up',
    'B.2 #12 Input validation warning': 'Outside training range',
    'B.2 #15 Model version footer': 'Model version',
    'B.2 #16 Contribute Data disabled': 'Contribute Data',
}
for label, needle in checks.items():
    assert needle in app_text, f'Missing: {label} (search string: {needle!r})'
print('Frontend validation passed. All B.2 detection strings present.')

### **Frontend Push**

**Process:** `upload_folder` ships `frontend_space_root/` to the frontend Space.

**Outcome:** Frontend Docker build kicks off; once live, the UI calls the backend via `BACKEND_URL`.

In [ ]:
#-------------
# FRONTEND PUSH
#-------------
upload_folder(folder_path='frontend_space_root', repo_id=FRONTEND_REPO, repo_type='space', commit_message='Pass 1 v2 frontend assets')
FRONTEND_URL = f'https://evagaiml-{SHORT_SLUG}-frontend.hf.space'
print(f'Frontend pushed: {FRONTEND_URL}')
print('Once both Docker builds complete, load BACKEND_URL first, then FRONTEND_URL.')

---

## **Expanded Executive Summary**

### TLDR

The Pass 1 rebuild produces three chronologically-respecting ensemble regressors (weekly, quarterly, yearly) for revenue forecasting across the Hill Country Grocer multi-banner network. Headline: weekly primary R-squared {{PASS_2: weekly_primary_r2}}, MAPE {{PASS_2: weekly_primary_mape}}\% on a held-out final 15\% of weeks. Primary deployment is the Streamlit frontend at {{PASS_2: frontend_url}}, backed by the FastAPI service at {{PASS_2: backend_url}}.

### Full Summary

**Objective:** Forecast Weekly_Revenue_USD per (product, store, week) for known products at known stores, with multi-horizon (weekly, quarterly, yearly) projection support and an interactive UI for store managers and category buyers.

**Data and Preparation:** 279,724 weekly rows across 11 stores, 3 banners, 4 regions, 300 UPCs, and 156 weeks. Imputed nulls (median for numerics, mode for categoricals, list-price backfill for missing promo prices). Kept zero-sales rows and outlier spikes (filtering both biases test metrics upward). Engineered Store_Age_Years, Promo_Discount_Pct, Is_On_Promo, Week_of_Year, Year. Dropped UPC and Item_Description as features (high-cardinality, no reusable signal).

**Iterative Development:** Six-candidate bake-off (Decision Tree, Bagging, Random Forest, Gradient Boosting, XGBoost, CatBoost) repeated independently for each horizon. Cross-validated grid search on training; ranked on validation; single-shot evaluation on the held-out test set. Pass 1 keeps grids modest to fit reference-notebook runtime; Pass 2 may expand where Pass 1 evidence suggests value.

**Performance Analysis:** Weekly primary {{PASS_2: weekly_primary_model_name}} achieves R-squared {{PASS_2: weekly_primary_r2}} / MAPE {{PASS_2: weekly_primary_mape}}\% on test. Quarterly primary {{PASS_2: quarterly_primary_model_name}}: R-squared {{PASS_2: quarterly_primary_r2}} / MAPE {{PASS_2: quarterly_primary_mape}}\%. Yearly primary {{PASS_2: yearly_primary_model_name}}: R-squared {{PASS_2: yearly_primary_r2}} / MAPE {{PASS_2: yearly_primary_mape}}\% — thin (one held-out year) and flagged as such in the frontend's conditional warning.

**Economic Impact:** Per-row weekly MAE of \${{PASS_2: weekly_test_mae}} translates to {{PASS_2: economic_impact_summary}} across the chain. Frontend exposes per-store totals and department breakdowns to make this actionable for store managers and category buyers.

**Deployment Readiness:** Two-Space deployment is live. Backend FastAPI serves `/v1/predict` (horizon + model routed), `/v1/models` (metadata), and `/v1/insights` (precomputed chain-wide insights). Frontend Streamlit covers single-prediction, multi-scenario stacking, chain-wide and selection-based insights, bulk CSV upload, and session-persistent inputs. Banner/region kept as model features but hidden from UI (auto-derived from Store_Number selection).

**Next Steps:** Pass 2 fills the placeholders (greppable via the PASS_2 token) from executed outputs and adds execution-revealed improvements (residual patterns, feature-importance surprises, hyperparameter expansion where Pass 1 evidence supports it). Out of scope for this rebuild: permanent data-add capability (deferred workstream), LLM-narrated insights (v1+ catalog feature), non-HuggingFace deployment targets.